In [3]:
import pandas as pd

# --- 1. LOAD WORLD BANK DATA ---
# (Using '../' to step out of the script folder first, skipping the top 4 junk lines)
df_gdp = pd.read_csv('../data/GDP per capita/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46.csv', skiprows=4)
df_women = pd.read_csv('../data/Internet Usage (Women)/API_IT.NET.USER.FE.ZS_DS2_en_csv_v2_16797/API_IT.NET.USER.FE.ZS_DS2_en_csv_v2_16797.csv', skiprows=4)
df_men = pd.read_csv('../data/Internet Usage (Men)/API_IT.NET.USER.MA.ZS_DS2_en_csv_v2_18514/API_IT.NET.USER.MA.ZS_DS2_en_csv_v2_18514.csv', skiprows=4)
df_total = pd.read_csv('../data/Internet Usage (Population)/API_IT.NET.USER.ZS_DS2_en_csv_v2_21/API_IT.NET.USER.ZS_DS2_en_csv_v2_21.csv', skiprows=4)

# --- 2. HELPER FUNCTION TO CLEAN AND MELT ---
def clean_and_melt(df, value_name):
    # Drop unnecessary columns and the phantom 'Unnamed: 68' column
    cols_to_drop = ['Indicator Name', 'Indicator Code']
    cols_to_drop += [c for c in df.columns if 'Unnamed' in c]
    df_clean = df.drop(columns=cols_to_drop, errors='ignore')
    
    # Melt the years into a single column
    df_long = pd.melt(df_clean, 
                      id_vars=['Country Name', 'Country Code'], 
                      var_name='Year', 
                      value_name=value_name)
    return df_long

# Apply the function to get our long-format tables
df_gdp_long = clean_and_melt(df_gdp, 'GDP_Per_Capita')
df_women_long = clean_and_melt(df_women, 'Female_Usage_Pct')
df_men_long = clean_and_melt(df_men, 'Male_Usage_Pct')
df_total_long = clean_and_melt(df_total, 'Internet_Usage_Pct')

# --- 3. MERGE WORLD BANK DATA & FILTER YEARS ---
clean_df = df_gdp_long.merge(df_women_long, on=['Country Name', 'Country Code', 'Year'])
clean_df = clean_df.merge(df_men_long, on=['Country Name', 'Country Code', 'Year'])
clean_df = clean_df.merge(df_total_long, on=['Country Name', 'Country Code', 'Year'])

# Convert Year to a number and keep only 2000 to 2023
clean_df['Year'] = pd.to_numeric(clean_df['Year'], errors='coerce')
clean_df = clean_df[(clean_df['Year'] >= 2000) & (clean_df['Year'] <= 2023)]

# --- 4. STANDARDIZE COUNTRY NAMES ---
# This ensures the World Bank names match the Kaggle names
country_fixes = {
    "Bahamas, The": "Bahamas",
    "Gambia, The": "Gambia",
    "Egypt, Arab Rep.": "Egypt",
    "Russian Federation": "Russia",
    "Korea, Rep.": "South Korea",
    "Iran, Islamic Rep.": "Iran",
    "Venezuela, RB": "Venezuela",
    "Turkiye": "Turkey",
    "Congo, Dem. Rep.": "Democratic Republic of Congo",
    "Congo, Rep.": "Republic of the Congo",
    "Syrian Arab Republic": "Syria",
    "Yemen, Rep.": "Yemen",
    "Slovak Republic": "Slovakia",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Lao PDR": "Laos",
    "Brunei Darussalam": "Brunei",
    "Hong Kong SAR, China": "Hong Kong",
    "Macao SAR, China": "Macao",
    "Turks and Caicos Islands": "Turks and Caicos"
}
clean_df['Country Name'] = clean_df['Country Name'].replace(country_fixes)

# --- 5. ADD THE PRICING & REGION DATA (FIXED) ---
# Read the Excel file
df_cost = pd.read_excel('../data/worldwide_mobile_data_pricing_data.xlsx', sheet_name='Historical Data')

# FIX 1: Attach the Region permanently to all years
# A country's region never changes, so we merge it using only the Country Name
regions_df = df_cost[['Name', 'Continental region']].drop_duplicates().rename(
    columns={'Name': 'Country Name', 'Continental region': 'Region'}
)
clean_df = clean_df.merge(regions_df, on='Country Name', how='left')

# FIX 2: Process the historical pricing
# Find all the columns that contain historical pricing
price_cols = [col for col in df_cost.columns if 'Average price of 1GB' in col and 'Global' not in col]

# Melt the pricing dataframe from Wide to Long
melted_pricing = pd.melt(
    df_cost,
    id_vars=['Name'], 
    value_vars=price_cols,
    var_name='Year_String',
    value_name='Cost_1GB_USD'
)

# Extract the 4-digit year from the column headers 
melted_pricing['Year'] = melted_pricing['Year_String'].str.extract(r'(\d{4})').astype(int)

# Clean up column names for merging
melted_pricing = melted_pricing.rename(columns={'Name': 'Country Name'})
melted_pricing = melted_pricing.drop(columns=['Year_String'])

# MERGE pricing on BOTH 'Country Name' and 'Year'
# We use Country Name here because the 2-letter vs 3-letter codes don't match!
clean_df = clean_df.merge(melted_pricing, on=['Country Name', 'Year'], how='left')

# --- 6. CLEAN NUMERICS AND CALCULATE AFFORDABILITY ---
# Strip out any '$' or ',' from the pricing data so Python can do math on it
clean_df['Cost_1GB_USD'] = clean_df['Cost_1GB_USD'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)

# Force columns to be numeric types
clean_df['Cost_1GB_USD'] = pd.to_numeric(clean_df['Cost_1GB_USD'], errors='coerce')
clean_df['GDP_Per_Capita'] = pd.to_numeric(clean_df['GDP_Per_Capita'], errors='coerce')

# Calculate the UN 2% Affordability Metric
clean_df['Affordability_Pct'] = ((clean_df['Cost_1GB_USD'] * 12) / clean_df['GDP_Per_Capita']) * 100

# --- 7. EXPORT THE FINAL FILE ---
# Drop completely empty rows so the dashboard doesn't process useless data
clean_df.dropna(subset=['GDP_Per_Capita', 'Internet_Usage_Pct'], how='all', inplace=True)

clean_df.to_csv('cleaned_dashboard_data.csv', index=False)
print("Data successfully cleaned and saved to 'cleaned_dashboard_data.csv'!")

Data successfully cleaned and saved to 'cleaned_dashboard_data.csv'!


In [2]:
# Calculate the percentage of missing values per column
missing_pct = clean_df.isna().mean() * 100

# Filter out columns with 0% missing, and sort the rest from highest to lowest
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

# Format the output to look like a clean percentage
print("Percentage of missing values per column:")
print(missing_pct.round(2).astype(str) + '%')

Percentage of missing values per column:
Region                100.0%
Cost_1GB_USD          100.0%
Affordability_Pct     100.0%
Female_Usage_Pct      76.89%
Male_Usage_Pct        76.88%
Internet_Usage_Pct    14.73%
GDP_Per_Capita         0.96%
dtype: object
